# Vision-IA: CNN propia desde cero

GPU NVIDIA, Drive autorizado, datos B train/val intactos y suplemento auditado. No extrae ni evalua test. Cualquier entrenamiento anterior queda conservado.


In [ ]:
from pathlib import Path
from google.colab import drive
import json, hashlib, zipfile, subprocess, sys, uuid
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Selecciona una GPU en Entorno de ejecucion.'
print(torch.cuda.get_device_name(0))
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Vision-IA')
EXPERIMENT='C2'
# Deja vacio para un entrenamiento nuevo; para reanudar, nombre exacto de su carpeta.
RESUME_RUN=''


In [ ]:
info=json.loads((BASE/f'automation/{EXPERIMENT}/package_ready.json').read_text())
assert info['initialization']=='random_from_scratch'
package=BASE/info['package']
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
assert digest(package)==info['sha256'], 'Drive no termino de sincronizar.'
PROJECT=Path('/content/vision_c2')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(package) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(PROJECT)
baseline=info['base_package']; source=BASE/baseline['package_relative_path']
assert digest(source)==baseline['package_sha256']
prefix='Vision-IA/data/experiments/B/processed/'
DATA=Path('/content/vision_data')/prefix.rstrip('/')
with zipfile.ZipFile(source) as archive:
    names=[n for n in archive.namelist() if n.startswith(prefix) and (n.startswith(prefix+'train/') or n.startswith(prefix+'val/') or n in (prefix+'manifest.csv',prefix+'split_summary.json'))]
    assert len(names)==4998
    for name in names:
        assert '..' not in Path(name).parts
        archive.extract(name,'/content/vision_data')
print('Train/val preparados. Test no extraido.')


In [ ]:
if RESUME_RUN:
    assert Path(RESUME_RUN).name==RESUME_RUN and RESUME_RUN.startswith(EXPERIMENT+'_')
    RUN=BASE/'results/C'/RESUME_RUN
    assert (RUN/'last_training.pth').is_file(), 'No existe estado para reanudar.'
else:
    run_name=EXPERIMENT+'_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%SZ_')+uuid.uuid4().hex[:8]
    RUN=BASE/'results/C'/run_name
    assert not RUN.exists()
command=[sys.executable,'-u','-m','scripts.train_compact','--config',info['config'],'--dataset-dir',str(DATA),'--output-dir',str(RUN)]
if RESUME_RUN:command.append('--resume')
print('ENTRENAMIENTO',RUN)
subprocess.run(command,cwd=PROJECT,check=True)


In [ ]:
# Selecciona usando solo val B; guarda publicacion con hashes para importar en Windows.
subprocess.run([sys.executable,'-u','-m','scripts.finalize_compact','--run-dir',str(RUN),'--dataset-dir',str(DATA)],cwd=PROJECT,check=True)
print('Resultados guardados:',RUN/'selected')
print('Windows: python -m scripts.import_compact --run-name',RUN.name)
